In [1]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow
import unicodedata



# Carregando variáveis de ambiente
load_dotenv()

True

In [2]:
# Variáveis de ambiente para autenticação e configuração do BigQuery
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
project_id = os.getenv("PROJECT_ID")

# Tabela SILVER no BigQuery
tabela_silver = os.getenv("SILVER_TABLE")

# Tabela GOLD no BigQuery
table_gold = os.getenv("GOLD_DISTRIBUIDORAS")

# Cria o cliente Bigquery ao projeto configurado
client = bigquery.Client(project=project_id)

In [3]:
# Inicializa o cliente do BigQuery usando credenciais de serviço
# O parâmetro 'project_id' especifica o projeto GCP onde as queries serão executadas
client = bigquery.Client.from_service_account_json(credencial, project=project_id)


query = f"""
SELECT *
FROM `{tabela_silver}`
"""

In [4]:
# Executa e converte pra DataFrame
resultado = client.query(query)
df = resultado.to_dataframe()

/var/data/python/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [5]:
df.head()

,regiao_sigla,estado_sigla,municipio,revenda,cnpj_da_revenda,nome_da_rua,numero_rua,complemento,bairro,cep,produto,data_da_coleta,valor_de_venda,valor_de_compra,unidade_de_medida,bandeira
0,ne,pe,jaboatao dos guararapes,posto dom helder ltda,03792455000185,rua doutor aniceto varejao,s/n,None,candeias,34420-000,diesel s10,27/01/2015,"2,64",None,r$ / litro,dislub
1,ne,pe,jaboatao dos guararapes,posto dom helder ltda,03792455000185,rua doutor aniceto varejao,s/n,None,candeias,34420-000,diesel s10,03/02/2015,"2,848","2,3856",r$ / litro,dislub
2,ne,pe,jaboatao dos guararapes,posto dom helder ltda,03792455000185,rua doutor aniceto varejao,s/n,None,candeias,34420-000,diesel s10,10/02/2015,"2,848",None,r$ / litro,dislub
3,ne,pe,jaboatao dos guararapes,posto dom helder ltda,03792455000185,rua doutor aniceto varejao,s/n,None,candeias,34420-000,diesel s10,18/02/2015,"2,848",None,r$ / litro,dislub
4,ne,pe,jaboatao dos guararapes,posto dom helder ltda,03792455000185,rua doutor aniceto varejao,s/n,None,candeias,34420-000,diesel s10,16/03/2015,"2,699","2,5406",r$ / litro,dislub


In [6]:
df_bandeira = df[["bandeira"]] \
    .drop_duplicates(subset="bandeira") \
    .reset_index(drop=True)

df_bandeira.insert(0, "bandeira_sk", range(1, len(df_bandeira) + 1))

In [7]:
df_bandeira

,bandeira_sk,bandeira
0,1,dislub
1,2,branca
2,3,petrobras distribuidora s.a.
3,4,vibra
4,5,raizen
5,6,ipiranga
6,7,setta distribuidora
7,8,federal
8,9,tdc distribuidora
9,10,temape


In [9]:
# Criação e carga da tabela Gold

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela Gold seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_bandeira,
    table_gold,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print("✅ Tabela Gold criada e dados carregados com sucesso")

✅ Tabela Gold criada e dados carregados com sucesso
